# Unit 4: Multi-Qubit Systems, Entanglement, & Adders

Plan 1.5 deliverable. Three parts:

1. Bell states on `qiskit_aer`
2. Multi-qubit gates and tensor products checked against NumPy
3. Half and full quantum adders (CNOT + Toffoli)

Qiskit uses little-endian ordering: basis index = `2*q1 + q0`, and bitstrings read `q1 q0`.

In [ ]:
from itertools import product

import numpy as np
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister, transpile
from qiskit.quantum_info import Operator, Statevector
from qiskit_aer import AerSimulator

sim = AerSimulator()

## Part 1: Bell states

Start from a 2-qubit basis state, apply H on q0, then CNOT (q0 → q1). Which Bell state you get depends on the input.

In [ ]:
SHOTS = 1024
S = 1 / np.sqrt(2)

# name: (X on q0?, X on q1?, expected statevector in |q1 q0> order)
BELL_STATES = {
    "Phi+": (0, 0, [S, 0, 0, S]),
    "Phi-": (1, 0, [S, 0, 0, -S]),
    "Psi+": (0, 1, [0, S, S, 0]),
    "Psi-": (1, 1, [0, S, -S, 0]),   # equal up to global phase
}


def bell_circuit(x0, x1):
    qc = QuantumCircuit(2)
    if x0:
        qc.x(0)
    if x1:
        qc.x(1)
    qc.h(0)
    qc.cx(0, 1)
    return qc


bell_circuit(0, 0).draw("mpl")

In [ ]:
bell_ok = True
for name, (x0, x1, expected) in BELL_STATES.items():
    qc = bell_circuit(x0, x1)

    sv_ok = Statevector(qc).equiv(Statevector(expected))

    meas = qc.copy()
    meas.measure_all()
    counts = sim.run(transpile(meas, sim), shots=SHOTS).result().get_counts()

    allowed = {k for k, amp in zip(["00", "01", "10", "11"], expected) if abs(amp) > 1e-9}
    only_allowed = set(counts) <= allowed
    balanced = all(0.4 * SHOTS <= counts.get(k, 0) <= 0.6 * SHOTS for k in allowed)

    ok = sv_ok and only_allowed and balanced
    bell_ok &= ok
    print(f"{name}: statevector={'OK' if sv_ok else 'FAIL'}  counts={dict(counts)}  -> {'PASS' if ok else 'FAIL'}")

print("\nall Bell state checks passed" if bell_ok else "\nSOME CHECKS FAILED")

In [ ]:
from qiskit.visualization import plot_histogram

demo = bell_circuit(0, 0)
demo.measure_all()
plot_histogram(sim.run(transpile(demo, sim), shots=SHOTS).result().get_counts())

## Part 2: Multi-qubit gates and tensor products

Compare Qiskit's `Operator` and `Statevector` against hand-written NumPy matrices. CNOT has control q0 and target q1.

In [ ]:
EXPECTED = {
    "CNOT": np.array([[1, 0, 0, 0],
                      [0, 0, 0, 1],
                      [0, 0, 1, 0],
                      [0, 1, 0, 0]], dtype=complex),
    "CZ": np.diag([1, 1, 1, -1]).astype(complex),
    "SWAP": np.array([[1, 0, 0, 0],
                      [0, 0, 1, 0],
                      [0, 1, 0, 0],
                      [0, 0, 0, 1]], dtype=complex),
}


def gate_circuit(name):
    qc = QuantumCircuit(2)
    {"CNOT": lambda: qc.cx(0, 1),
     "CZ": lambda: qc.cz(0, 1),
     "SWAP": lambda: qc.swap(0, 1)}[name]()
    return qc


def check(label, condition):
    print(f"{'PASS' if condition else 'FAIL'}: {label}")
    return bool(condition)

In [ ]:
gates_ok = True
for name, matrix in EXPECTED.items():
    qc = gate_circuit(name)
    op = Operator(qc).data
    gates_ok &= check(f"{name} matrix matches hand-written matrix", np.allclose(op, matrix))
    gates_ok &= check(f"{name} is unitary (U†U = I)", np.allclose(op.conj().T @ op, np.eye(4)))

    for idx in range(4):
        basis = np.zeros(4, dtype=complex)
        basis[idx] = 1
        out = Statevector(basis).evolve(qc).data
        gates_ok &= check(f"{name} on |{idx:02b}> matches column {idx}", np.allclose(out, matrix[:, idx]))

In [ ]:
# Tensor product: |1> on q1, |+> on q0
plus = np.array([1, 1]) / np.sqrt(2)
one = np.array([0, 1])

prod_qc = QuantumCircuit(2)
prod_qc.h(0)
prod_qc.x(1)

gates_ok &= check("np.kron(|1>, |+>) equals Qiskit statevector of H(q0), X(q1)",
                  np.allclose(np.kron(one, plus), Statevector(prod_qc).data))

# Separability: a product state reshapes to a rank-1 2x2 matrix, a Bell state to rank 2
rank_product = np.linalg.matrix_rank(Statevector(prod_qc).data.reshape(2, 2))
rank_bell = np.linalg.matrix_rank(Statevector(bell_circuit(0, 0)).data.reshape(2, 2))
gates_ok &= check("product state is separable (rank 1)", rank_product == 1)
gates_ok &= check("Bell state is not separable (rank 2)", rank_bell == 2)

print("\nall checks passed" if gates_ok else "\nSOME CHECKS FAILED")

## Part 3: Quantum adders

- **Half adder:** `q0 = a`, `q1 = b`, `q2 = 0` → `q1 = sum`, `q2 = carry`
- **Full adder:** `q0 = a`, `q1 = b`, `q2 = cin`, `q3 = 0` → `q2 = sum`, `q3 = carry-out`

Results are read as the bitstring `carry sum`.

In [ ]:
def half_adder(a, b):
    q = QuantumRegister(3, "q")
    c = ClassicalRegister(2, "c")
    qc = QuantumCircuit(q, c)
    for i, bit in enumerate((a, b)):
        if bit:
            qc.x(i)
    qc.ccx(0, 1, 2)   # carry = a AND b
    qc.cx(0, 1)       # q1 = a XOR b (sum)
    qc.measure(1, 0)  # c0 = sum
    qc.measure(2, 1)  # c1 = carry
    return qc


def full_adder(a, b, cin):
    q = QuantumRegister(4, "q")
    c = ClassicalRegister(2, "c")
    qc = QuantumCircuit(q, c)
    for i, bit in enumerate((a, b, cin)):
        if bit:
            qc.x(i)
    qc.ccx(0, 1, 3)   # cout = a AND b
    qc.cx(0, 1)       # q1 = a XOR b
    qc.ccx(1, 2, 3)   # cout ^= (a XOR b) AND cin
    qc.cx(1, 2)       # q2 = a XOR b XOR cin (sum)
    qc.measure(2, 0)  # c0 = sum
    qc.measure(3, 1)  # c1 = carry-out
    return qc


def run_adder(qc, shots=100):
    counts = sim.run(transpile(qc, sim), shots=shots).result().get_counts()
    assert len(counts) == 1, f"expected a deterministic result, got {counts}"
    return next(iter(counts))


full_adder(1, 1, 1).draw("mpl")

In [ ]:
adder_ok = True

print("Half adder (a, b -> carry, sum)")
for a, b in product((0, 1), repeat=2):
    total = a + b
    expected = f"{total // 2}{total % 2}"
    got = run_adder(half_adder(a, b))
    good = got == expected
    adder_ok &= good
    print(f"  a={a} b={b} -> carry,sum = {got}  expected {expected}  {'PASS' if good else 'FAIL'}")

print("\nFull adder (a, b, cin -> cout, sum)")
for a, b, cin in product((0, 1), repeat=3):
    total = a + b + cin
    expected = f"{total // 2}{total % 2}"
    got = run_adder(full_adder(a, b, cin))
    good = got == expected
    adder_ok &= good
    print(f"  a={a} b={b} cin={cin} -> cout,sum = {got}  expected {expected}  {'PASS' if good else 'FAIL'}")

print("\nall adder checks passed" if adder_ok else "\nSOME CHECKS FAILED")